# Research Copilot - Full GPU Training Pipeline
**Course**: B.Tech AI & Data Science Viva Project  
**Hardware Target**: Google Colab T4 GPU (Runtime -> Change runtime type -> T4 GPU)

This notebook trains:
1. `train_embedder.py --mode full`: Contrastive embedding fine-tuning with hard negatives and BEIR SciFact out-of-domain evaluation.
2. `train_scibert.py --mode full`: SciBERT fine-tuning with Early Stopping on the validation split.
3. `train_gbdt.py`: Native LightGBM and XGBoost tree classifiers on the exact date-boundary splits.

Trained checkpoints, metrics, and logs are automatically pushed back to your private Hugging Face repository upon completion.

In [ ]:
# 1. Verify NVIDIA GPU Hardware & Drivers
!nvidia-smi

In [ ]:
# 2. Authenticate securely using Colab Secrets or hidden prompt
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Enter your Hugging Face write token (typing hidden): ')

os.environ['HF_TOKEN'] = HF_TOKEN
print('Hugging Face authentication token configured successfully.')

In [ ]:
# 3. Download project dataset and training scripts from your private repo
!pip install -q huggingface_hub
from huggingface_hub import HfApi, snapshot_download

api = HfApi(token=HF_TOKEN)
username = api.whoami()['name']
repo_id = f'{username}/research-copilot-training'
os.environ['HF_REPO_ID'] = repo_id
print(f'Syncing dataset and scripts from private repo: {repo_id}...')

snapshot_download(repo_id=repo_id, repo_type='dataset', local_dir='.', token=HF_TOKEN)
print('Workspace synced successfully!')

In [ ]:
# 4. Run the full training pipeline (Embedder + SciBERT + LightGBM/XGBoost)
!python colab_run.py

In [ ]:
# 5. Display Summary Metrics and Proof of Training
import json
print('='*70)
print('                    FINAL GPU METRICS SUMMARY')
print('='*70)
if os.path.exists('data/scibert_metrics.json'):
    print('\n--- SciBERT Metrics ---')
    print(open('data/scibert_metrics.json').read())

if os.path.exists('data/gbdt_metrics.json'):
    print('\n--- LightGBM & XGBoost Metrics ---')
    print(open('data/gbdt_metrics.json').read())

if os.path.exists('results/search_benchmark.md'):
    print('\n--- Search Benchmark Report Summary ---')
    lines = open('results/search_benchmark.md').readlines()
    print(''.join(lines[:45]))